In [23]:
import json
from pathlib import Path
from imgnet.collections.store import IndexedDatasets
from imgnet.collections.source import ZenodoSource
from imgtools.nifti.crawl import Crawler
from rich import print
import pandas as pd

# STS-Tooth

In [24]:
DATASET_NAME = "STS-Tooth"

indexed_datasets = IndexedDatasets("indexed_datasets") # This will download the latest version of the indexed_datasets if you dont provide a path
print(indexed_datasets.collections[:10])



15:52:22 [info     ] Indexed datasets path: /home/gpudual/bhklab/declan/med-image_index/indexed_datasets [imgnet] call=store.__init__:65


[
    '4D-Lung',
    'ACRIN-6698',
    'ACRIN-Contralateral-Breast-MR',
    'ACRIN-FLT-Breast',
    'ACRIN-NSCLC-FDG-PET',
    'AMOS',
    'Abdomen Atlas 3',
    'Adrenal-ACC-Ki67-Seg',
    'Advanced-MRI-Breast-Lesions',
    'Anti-PD-1_Lung'
]

In [25]:
description = """
In response to the increasing prevalence of dental diseases, dental health, a vital aspect of human well-being, warrants greater attention.

Panoramic X-ray images (PXI) and Cone Beam Computed Tomography (CBCT) are key tools for dentists in diagnosing and treating dental conditions.

Additionally, deep learning for tooth segmentation can focus on relevant treatment information and localize lesions.

However, the scarcity of publicly available PXI and CBCT datasets hampers their use in tooth segmentation tasks.

Therefore, this paper presents a multimodal dataset for semi-supervised deep learning in dental PXI and CBCT, named STS-2D-Tooth and STS-3D-Tooth.

STS-2D-Tooth includes 4,000 images and 900 masks, categorized by age into children and adults.

Moreover, we have collected CBCTs providing more detailed and three-dimensional information, resulting in the STS-3D-Tooth dataset comprising 148,400 unlabeled scans and 8,800 masks.

To our knowledge, this is the first multimodal dataset combining dental PXI and CBCT, and it is the largest tooth segmentation dataset, a significant step forward for the advancement of tooth segmentation.


"""

In [26]:
source = ZenodoSource(
    file_type="nifti",
    source="zenodo",
    record_id="10597292",
    description=description,
    post_download=["unzip"]
)

In [27]:
dataset_index_path = (indexed_datasets.path / DATASET_NAME)
dataset_index_path.mkdir(parents=True, exist_ok=True)
(dataset_index_path / "parquet").mkdir(exist_ok=True)
with open(dataset_index_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)

In [28]:

print(indexed_datasets.source_config(DATASET_NAME))

ZenodoSource(
    file_type=<FileType.NIFTI: 'nifti'>,
    post_download=['unzip'],
    source='zenodo',
    record_id='10597292',
    filenames=None,
    description='\nIn response to the increasing prevalence of dental diseases, dental health, a vital aspect of 
human well-being, warrants greater attention.\n\nPanoramic X-ray images (PXI) and Cone Beam Computed Tomography 
(CBCT) are key tools for dentists in diagnosing and treating dental conditions.\n\nAdditionally, deep learning for 
tooth segmentation can focus on relevant treatment information and localize lesions.\n\nHowever, the scarcity of 
publicly available PXI and CBCT datasets hampers their use in tooth segmentation tasks.\n\nTherefore, this paper 
presents a multimodal dataset for semi-supervised deep learning in dental PXI and CBCT, named STS-2D-Tooth and 
STS-3D-Tooth.\n\nSTS-2D-Tooth includes 4,000 images and 900 masks, categorized by age into children and 
adults.\n\nMoreover, we have collected CBCTs providing more detailed and three-dimensional information, resulting 
in the STS-3D-Tooth dataset comprising 148,400 unlabeled scans and 8,800 masks.\n\nTo our knowledge, this is the 
first multimodal dataset combining dental PXI and CBCT, and it is the largest tooth segmentation dataset, a 
significant step forward for the advancement of tooth segmentation.\n\n\n'
)

In [29]:
temp_data_path = Path("temp_data") / DATASET_NAME
temp_data_path.mkdir(parents=True, exist_ok=True)
#downloader = indexed_datasets.downloader(DATASET_NAME).download(temp_data_path)

In [30]:
# import zipfile
# from glob import glob
# import os


# def extract_multipart_zip(first_part, extract_to=None):
#     """
#     Combine .zip.001, .zip.002, ... parts and extract the resulting ZIP.
#     """
#     if extract_to is None:
#         extract_to = os.path.dirname(first_part)

#     os.makedirs(extract_to, exist_ok=True)

#     # Remove .zip.001
#     base = first_part.removesuffix(".001")

#     # Find all parts in order
#     parts = sorted(glob(f"{base}.*"))

#     combined_zip = os.path.join(
#         os.path.dirname(first_part),
#         os.path.basename(base),
#     )

#     print(f"Combining {len(parts)} parts...")

#     with open(combined_zip, "wb") as outfile:
#         for part in parts:
#             print(f"  Adding {os.path.basename(part)}")
#             with open(part, "rb") as infile:
#                 outfile.write(infile.read())

#     # Extract combined ZIP
#     with zipfile.ZipFile(combined_zip, "r") as zip_ref:
#         zip_ref.extractall(extract_to)

#     # Remove temporary combined ZIP
#     os.remove(combined_zip)

#     print(f"Extracted to {extract_to}")


# for zip_file in glob(f"{temp_data_path}/*.zip.001"):
#     print(zip_file)

#     zip_name = os.path.basename(zip_file).removesuffix(".zip.001")
#     extract_to = os.path.join(temp_data_path)

#     extract_multipart_zip(zip_file, extract_to)

In [31]:
crawler = Crawler(
    nifti_dir=temp_data_path,
    scan_name_pattern="SD-Tooth/STS-3D-Tooth/{ROI_Type}/{isLabeled}/Image/{reference_id}.nii.gz",
    mask_name_pattern="SD-Tooth/STS-3D-Tooth/{ROI_Type}/{isLabeled}/Mask/{reference_id}.nii.gz",
    output_dir=indexed_datasets.path,
    n_jobs=10,
    force=True,
    deep=False,
)
crawler.crawl()


In [32]:
index_df = pd.read_csv(indexed_datasets.path / DATASET_NAME / "index.csv")
index_df

,ROI_Type,isLabeled,reference_id,filepath,file_type,reference_scan
0,Integrity,Labeled,Integrity_Labeled_Integrity_L_001,SD-Tooth/STS-3D-Tooth/Integrity/Labeled/Image/...,scan,NaN
1,Integrity,Labeled,Integrity_Labeled_Integrity_L_002,SD-Tooth/STS-3D-Tooth/Integrity/Labeled/Image/...,scan,NaN
2,Integrity,Labeled,Integrity_Labeled_Integrity_L_003,SD-Tooth/STS-3D-Tooth/Integrity/Labeled/Image/...,scan,NaN
3,Integrity,Labeled,Integrity_Labeled_Integrity_L_004,SD-Tooth/STS-3D-Tooth/Integrity/Labeled/Image/...,scan,NaN
4,Integrity,Labeled,Integrity_Labeled_Integrity_L_005,SD-Tooth/STS-3D-Tooth/Integrity/Labeled/Image/...,scan,NaN
...,...,...,...,...,...,...
398,ROI,Unlabeled,ROI_Unlabeled_ROI_U_104,SD-Tooth/STS-3D-Tooth/ROI/Unlabeled/Image/ROI_...,scan,NaN
399,ROI,Unlabeled,ROI_Unlabeled_ROI_U_105,SD-Tooth/STS-3D-Tooth/ROI/Unlabeled/Image/ROI_...,scan,NaN
400,ROI,Unlabeled,ROI_Unlabeled_ROI_U_106,SD-Tooth/STS-3D-Tooth/ROI/Unlabeled/Image/ROI_...,scan,NaN
401,ROI,Unlabeled,ROI_Unlabeled_ROI_U_107,SD-Tooth/STS-3D-Tooth/ROI/Unlabeled/Image/ROI_...,scan,NaN


In [33]:
def map_modality(row):
    if row["file_type"] == "scan":
            return "CT"
    else:
        return "SEG"

def map_bodypart(row):
    if row["ROI_Type"] == "Integrity":
            return "HEAD"
    else:
        return "JAW"
    

index_df["Modality"] = index_df.apply(map_modality, axis=1)
index_df["BodyPartExamined"] = index_df.apply(map_bodypart, axis=1)

index_df["ROINames"] = "TOOTH"

index_df.drop(columns=["ROI_Type", "isLabeled"], inplace=True)
index_df

,reference_id,filepath,file_type,reference_scan,Modality,BodyPartExamined,ROINames
0,Integrity_Labeled_Integrity_L_001,SD-Tooth/STS-3D-Tooth/Integrity/Labeled/Image/...,scan,NaN,CT,HEAD,TOOTH
1,Integrity_Labeled_Integrity_L_002,SD-Tooth/STS-3D-Tooth/Integrity/Labeled/Image/...,scan,NaN,CT,HEAD,TOOTH
2,Integrity_Labeled_Integrity_L_003,SD-Tooth/STS-3D-Tooth/Integrity/Labeled/Image/...,scan,NaN,CT,HEAD,TOOTH
3,Integrity_Labeled_Integrity_L_004,SD-Tooth/STS-3D-Tooth/Integrity/Labeled/Image/...,scan,NaN,CT,HEAD,TOOTH
4,Integrity_Labeled_Integrity_L_005,SD-Tooth/STS-3D-Tooth/Integrity/Labeled/Image/...,scan,NaN,CT,HEAD,TOOTH
...,...,...,...,...,...,...,...
398,ROI_Unlabeled_ROI_U_104,SD-Tooth/STS-3D-Tooth/ROI/Unlabeled/Image/ROI_...,scan,NaN,CT,JAW,TOOTH
399,ROI_Unlabeled_ROI_U_105,SD-Tooth/STS-3D-Tooth/ROI/Unlabeled/Image/ROI_...,scan,NaN,CT,JAW,TOOTH
400,ROI_Unlabeled_ROI_U_106,SD-Tooth/STS-3D-Tooth/ROI/Unlabeled/Image/ROI_...,scan,NaN,CT,JAW,TOOTH
401,ROI_Unlabeled_ROI_U_107,SD-Tooth/STS-3D-Tooth/ROI/Unlabeled/Image/ROI_...,scan,NaN,CT,JAW,TOOTH


In [34]:
from imgnet.collections.store import IndexedDatasets
from imgindex.model import validate_index
from imgindex.parquet import csv_to_parquet


index_df.rename(columns={"reference_id": "SampleID"}, inplace=True)
index_df["SampleID"] = index_df["SampleID"].astype(str)
index_df.to_csv(indexed_datasets.path / DATASET_NAME / "index.csv", index=False)
csv_to_parquet(index_df, indexed_datasets.path / DATASET_NAME)
validate_index(index_df, "nifti", lazy=True)

,SampleID,filepath,file_type,reference_scan,Modality,BodyPartExamined,ROINames
0,Integrity_Labeled_Integrity_L_001,SD-Tooth/STS-3D-Tooth/Integrity/Labeled/Image/...,scan,NaN,CT,HEAD,TOOTH
1,Integrity_Labeled_Integrity_L_002,SD-Tooth/STS-3D-Tooth/Integrity/Labeled/Image/...,scan,NaN,CT,HEAD,TOOTH
2,Integrity_Labeled_Integrity_L_003,SD-Tooth/STS-3D-Tooth/Integrity/Labeled/Image/...,scan,NaN,CT,HEAD,TOOTH
3,Integrity_Labeled_Integrity_L_004,SD-Tooth/STS-3D-Tooth/Integrity/Labeled/Image/...,scan,NaN,CT,HEAD,TOOTH
4,Integrity_Labeled_Integrity_L_005,SD-Tooth/STS-3D-Tooth/Integrity/Labeled/Image/...,scan,NaN,CT,HEAD,TOOTH
...,...,...,...,...,...,...,...
398,ROI_Unlabeled_ROI_U_104,SD-Tooth/STS-3D-Tooth/ROI/Unlabeled/Image/ROI_...,scan,NaN,CT,JAW,TOOTH
399,ROI_Unlabeled_ROI_U_105,SD-Tooth/STS-3D-Tooth/ROI/Unlabeled/Image/ROI_...,scan,NaN,CT,JAW,TOOTH
400,ROI_Unlabeled_ROI_U_106,SD-Tooth/STS-3D-Tooth/ROI/Unlabeled/Image/ROI_...,scan,NaN,CT,JAW,TOOTH
401,ROI_Unlabeled_ROI_U_107,SD-Tooth/STS-3D-Tooth/ROI/Unlabeled/Image/ROI_...,scan,NaN,CT,JAW,TOOTH
